# R05 — N01/R04 Frozen SSL Feature Bank (Seed 42)

**Kaggle notebook:** `lesion-r05-n01-dual-ssl-featurebank-s42`  
**Required notebook outputs:** completed `lesion-n01-dataset-audit-grouped-splits`, `lesion-r00-protocol-lock`, `lesion-r04-ssl-classification-s42`.  
**GPU:** ON; **Internet:** OFF. No retraining. Extract 2,048-dim frozen Rotation and SimCLR R04 features from train, validation, and meta_train only. Test set never accessed.

This uses the earlier **N01** lesion-aware cohort (not HF00 or O01). R04 fine-tuned encoder weights are used, not raw R02/R03 encoders.

## Configuration, libraries, and deterministic execution

In [1]:
import os,sys,json,time,hashlib,random,zipfile,shutil,platform,io,math
from pathlib import Path
from collections import defaultdict
import numpy as np,pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
from torchvision.models import resnet50

NAME='lesion-r05-n01-dual-ssl-featurebank-s42'
SEED=42
INPUT=Path(os.getenv('R05_INPUT_DIR','/kaggle/input' if Path('/kaggle/input').exists() else '/mnt/data/r05_test_inputs')).resolve()
OUT=Path(os.getenv('R05_OUT','/kaggle/working/'+NAME if Path('/kaggle/working').exists() else './'+NAME)).resolve()
STAGING=Path(os.getenv('R05_STAGE_DIR','/kaggle/working/.r05_inputs' if Path('/kaggle/working').exists() else './.r05_inputs')).resolve()
BATCH=int(os.getenv('R05_BATCH','48'))
WORKERS=int(os.getenv('R05_WORKERS','2'))
AMP=os.getenv('R05_AMP','1')=='1'
MAX_HOURS=8.0
START=time.monotonic()
CLASSES=['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
EXPECTED={'train':16490,'validation':2186,'meta_train':2267}
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type!='cuda' and os.getenv('R05_ALLOW_CPU','0')!='1':raise RuntimeError('Enable Kaggle GPU (CPU permitted only for local R05_ALLOW_CPU=1 smoke test).')
assert BATCH>=1 and WORKERS>=0
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
OUT.mkdir(parents=True,exist_ok=True)
for n in ['features','manifest','reports','tables']:(OUT/n).mkdir(exist_ok=True)

def req(v,msg):
    if not bool(v):raise RuntimeError(msg)
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(2**20),b''):h.update(b)
    return h.hexdigest()
def jsonwrite(p,obj):
    p=Path(p);p.parent.mkdir(exist_ok=True,parents=True)
    q=p.with_suffix(p.suffix+'.tmp');q.write_text(json.dumps(obj,indent=2,allow_nan=False,default=str));q.replace(p)
print(NAME,torch.__version__,'device=',DEVICE,'input=',INPUT)


lesion-r05-n01-dual-ssl-featurebank-s42 2.11.0+cu130 device= cuda input= /kaggle/input


## Resolve real Kaggle output folders and _output_.zip archives before loading GPUs

In [2]:
def find_roots(base,marker,kind):
    found=[]
    for p in base.rglob(Path(marker).name):
        if not p.is_file() or not p.as_posix().endswith(marker):continue
        r=p
        for _ in Path(marker).parts:r=r.parent
        try:o=json.loads(p.read_text())
        except Exception:continue
        if kind=='N01' and o.get('project')!='lesion-n01-dataset-audit-grouped-splits':continue
        if kind=='R00' and o.get('name')!='lesion-r00-protocol-lock':continue
        if kind=='R04' and (o.get('stage')!='R04' or o.get('seed')!=SEED):continue
        found.append(r.resolve())
    return sorted(set(found))

def safe_unpack(archive,target):
    target.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        infos=[x for x in z.infolist() if not x.is_dir()]
        total=sum(x.file_size for x in infos)
        free=shutil.disk_usage(target).free
        req(total < min(15*2**30,int(free*.90)),f'Insufficient disk for {archive.name}: need {total/2**30:.2f} GiB')
        for member in infos:
            p=Path(member.filename.replace('\\','/'))
            req(not p.is_absolute() and '..' not in p.parts,f'Unsafe ZIP member {member.filename}')
            dest=(target/p).resolve()
            req(dest.is_relative_to(target.resolve()),'ZIP traversal rejected')
            dest.parent.mkdir(parents=True,exist_ok=True)
            if dest.is_file() and dest.stat().st_size==member.file_size:continue
            with z.open(member) as a,open(dest,'wb') as b:shutil.copyfileobj(a,b,1024*1024)
    return target

SPECS={'N01':('reports/protocol.json','project'),'R00':('configs/experiment_protocol.json','name'),'R04':('reports/run_summary.json','stage')}
roots={k:find_roots(INPUT,v[0],k) for k,v in SPECS.items()}
if any(len(v)!=1 for v in roots.values()):
    archives=sorted(p for p in INPUT.rglob('*.zip') if p.is_file())
    # Kaggle notebook outputs are commonly mounted as _output_.zip, sometimes with another ZIP inside.
    for level in range(2):
        if all(len(v)==1 for v in roots.values()):break
        changed=False
        for archive in archives:
            marker=hashlib.sha256(str(archive).encode()).hexdigest()[:12]
            dest=STAGING/f'archive_{marker}'
            flag=dest/'.extracted.ok'
            if flag.exists():continue
            with zipfile.ZipFile(archive) as z:
                names=z.namelist()
                relevant=archive.name=='_output_.zip' or any(any(n.endswith(spec[0]) for spec in SPECS.values()) for n in names)
                relevant|=any(n.lower().endswith('.zip') and any(n.lower().endswith('.zip') for n in names) for n in names) and level==0
            if not relevant:continue
            print('Preparing Kaggle archive:',archive)
            safe_unpack(archive,dest);flag.write_text('ok');changed=True
        for k,(marker,_) in SPECS.items():
            if len(roots[k])!=1:roots[k]=find_roots(INPUT,marker,k)+find_roots(STAGING,marker,k)
            roots[k]=sorted(set(roots[k]))
        archives=sorted(set(archives+[p for p in STAGING.rglob('*.zip') if p.is_file()]))
        if not changed:break
for kind,paths in roots.items():
    req(len(paths)==1,f'{kind}: expected one complete notebook output with marker {SPECS[kind][0]}, found {paths}. Attach N01, R00, and completed R04 seed42 outputs.')
N01,R00,R04=(roots[k][0] for k in ['N01','R00','R04'])
print('Resolved:',{'N01':str(N01),'R00':str(R00),'R04':str(R04)})


Resolved: {'N01': '/kaggle/input/notebooks/rinkyranisaha/lesion-n01/lesion-n01-dataset-audit-grouped-splits', 'R00': '/kaggle/input/notebooks/sudiptoroy007/lesion-r00-protocol-lock/lesion-r00-protocol-lock', 'R04': '/kaggle/input/notebooks/sudiptoroy007/lesion-r04-ssl-classification-s42/lesion-r04-ssl-classification-s42'}


## Verify exact N01/R00 lesion-disjoint manifests and R04 checkpoint provenance

In [3]:
protocol=json.loads((R00/'configs/experiment_protocol.json').read_text())
checks=json.loads((R00/'reports/artifact_checksums.json').read_text())
n01=json.loads((N01/'reports/protocol.json').read_text())
req(protocol['status']=='LOCKED' and protocol['classes']==CLASSES,'Wrong R00 protocol/classes')
req(protocol['data_access']['ssl_finetuning']==['train'],'R00 data access changed')
req(protocol['data_access']['ensemble_meta_fit']==['meta_train'],'R00 meta_train role changed')
req(sha(R00/'configs/experiment_protocol.json')==checks['configs/experiment_protocol.json'],'R00 protocol SHA mismatch')
manifest_hash=protocol['n01_manifest_sha256']
req(sha(N01/'splits/experiment_master_manifest.csv')==manifest_hash,'N01 manifest SHA mismatch')
req(n01.get('manifest_sha256')==manifest_hash,'N01/R00 mismatch')
frames={}
for split in ['train','validation','meta_train']:
    fp=R00/'splits'/f'{split}.csv'
    req(sha(fp)==checks[f'splits/{split}.csv'],f'R00 split SHA mismatch: {split}')
    f=pd.read_csv(fp,dtype='string',keep_default_na=False).sort_values('image_id').reset_index(drop=True)
    req(len(f)==EXPECTED[split] and f.image_id.is_unique,f'Unexpected {split} counts or duplicated IDs')
    req(f.split.eq(split).all() and f.label.isin(CLASSES).all(),'Wrong split or class')
    req(f.class_index.eq(f.label.map({n:str(i) for i,n in enumerate(CLASSES)})).all(),'Wrong class index')
    for s in f.image_relpath:
        p=Path(s)
        req(not p.is_absolute() and '..' not in p.parts and p.parts[:1]==('images',),f'Unsafe image path: {s}')
        req((N01/p).is_file(),f'Missing N01 processed image: {s}')
    frames[split]=f
for a,b in [('train','validation'),('train','meta_train'),('validation','meta_train')]:
    for k in ['image_id','group_id','source_sha256']:
        req(set(frames[a][k]).isdisjoint(set(frames[b][k])),f'{k} overlap: {a} vs {b}')

run=json.loads((R04/'reports/run_summary.json').read_text())
req(run.get('status')=='COMPLETED' and run.get('seed')==SEED,'R04 not completed for seed 42')
req(run['n01_manifest_sha256']==manifest_hash,'R04/N01 mismatch')
encoders={}; heads={};orig_val={};provenance={}
for source in ['rotation','simclr']:
    epath=R04/'models'/source/'encoder_best.pt'
    bpath=R04/'models'/source/'best.pt'
    ppath=R04/'predictions'/source/'validation.npz'
    mpath=R04/'reports'/source/'final_metrics.json'
    cpath=R04/'reports'/source/'config.json'
    req(all(p.is_file() for p in [epath,bpath,ppath,mpath,cpath]),f'Missing R04 {source} checkpoint/prediction files')
    enc=torch.load(epath,map_location='cpu',weights_only=False)
    best=torch.load(bpath,map_location='cpu',weights_only=False)
    meta=json.loads(mpath.read_text());cfg=json.loads(cpath.read_text())
    req(meta['status']=='COMPLETED' and meta['source']==source and meta['seed']==SEED,'Wrong R04 model')
    req(enc.get('stage')=='R04' and enc.get('source')==source and enc.get('seed')==SEED,'Wrong encoder metadata')
    req(enc['n01_manifest_sha256']==manifest_hash,'Wrong encoder N01 identity')
    req(enc['r00_protocol_sha256']==checks['configs/experiment_protocol.json'],'Wrong encoder R00 identity')
    req(enc['signature']==cfg['config_sha256']==best['signature'],'R04 checkpoint signature mismatch')
    req(enc['feature_dim']==2048 and enc['fc']=='Identity','Wrong encoder feature size')
    req(meta['n01_manifest_sha256']==manifest_hash and best['n01_manifest_sha256']==manifest_hash,'R04 N01 mismatch')
    req(all(torch.equal(v.cpu(),best['model']['encoder.'+k].cpu()) for k,v in enc['encoder_state'].items()),'R04 exported encoder does not equal selected best classifier')
    with np.load(ppath,allow_pickle=False) as z:
        req(list(z['classes'].astype(str))==CLASSES,'R04 class order mismatch')
        ids=z['image_id'].astype(str);y=z['y'].astype(int);prob=z['proba'].astype(float)
        req(np.array_equal(ids,frames['validation'].image_id.astype(str).to_numpy()),'R04 val image order mismatch')
        req(np.array_equal(y,frames['validation'].class_index.astype(int).to_numpy()),'R04 val label mismatch')
        req(prob.shape==(EXPECTED['validation'],8) and np.isfinite(prob).all(),'Wrong R04 validation probabilities')
        req(np.max(abs(prob.sum(1)-1))<1e-3,'R04 probabilities not normalized')
        orig_val[source]=prob
    encoders[source]=enc['encoder_state']
    heads[source]={'weight':best['model']['classifier.weight'],'bias':best['model']['classifier.bias']}
    provenance[source]={'encoder_sha256':sha(epath),'best_classifier_sha256':sha(bpath),'validation_prob_sha256':sha(ppath),'best_epoch':meta['best_finetune_epoch'],'val_bacc':meta['validation_metrics']['balanced_accuracy']}
print('PRE-FLIGHT PASS — group-disjoint N01 16490 / 2186 / 2267; exact R04 encoders and validation identities verified')
print('R04 reference BACC:',{k:round(v['val_bacc']*100,3) for k,v in provenance.items()})


PRE-FLIGHT PASS — group-disjoint N01 16490 / 2186 / 2267; exact R04 encoders and validation identities verified
R04 reference BACC: {'rotation': 57.717, 'simclr': 57.562}


## Compare reconstructed R04 predictions before extracting feature bank

In [4]:
NORM=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
class Images(Dataset):
    def __init__(self,f):self.f=f.reset_index(drop=True)
    def __len__(self):return len(self.f)
    def __getitem__(self,i):
        with Image.open(N01/Path(self.f.image_relpath.iloc[i])) as im:x=NORM(im.convert('RGB'))
        return x

class FullR04(nn.Module):
    def __init__(self,src):
        super().__init__()
        self.encoder=resnet50(weights=None);self.encoder.fc=nn.Identity()
        self.encoder.load_state_dict(encoders[src],strict=True)
        self.classifier=nn.Linear(2048,8)
        self.classifier.load_state_dict(heads[src],strict=True)
    def forward(self,x):return self.classifier(self.encoder(x))

models={src:FullR04(src).to(DEVICE).eval() for src in ['rotation','simclr']}
subset=frames['validation'].iloc[:min(12,len(frames['validation']))]
xs=torch.stack([Images(subset)[i] for i in range(len(subset))]).to(DEVICE)
with torch.inference_mode():
    for source,model in models.items():
        with torch.autocast(device_type=DEVICE.type,enabled=AMP and DEVICE.type=='cuda'):
            pred=torch.softmax(model(xs).float(),dim=1).cpu().numpy()
        err=float(np.max(abs(pred-orig_val[source][:len(subset)])))
        req(err<.025,f'{source} classifier reconstructed probabilities disagree with R04 (max diff={err:.4g}). Stop before GPU feature extraction.')
        print('R04 reconstruction',source,'max probability difference',round(err,6))
for model in models.values():
    model.classifier=nn.Identity()
    req(isinstance(model.classifier,nn.Identity),'Classifier removal failed')
print('Both feature encoders verified, classifier heads removed.')
# Optional resumability: previous R05 notebook output can be attached as R05_PREVIOUS_ROOT.
previous=os.getenv('R05_PREVIOUS_ROOT','').strip()
if previous:
    prev=Path(previous).expanduser().resolve()
    if prev.is_file() and zipfile.is_zipfile(prev):
        target=STAGING/'previous_r05';safe_unpack(prev,target)
        choices=[p.parent.parent for p in target.rglob('final_status.json') if p.parent.name=='reports']
        req(len(choices)==1,'Previous R05 ZIP must contain exactly one status report')
        prev=choices[0]
    prior=json.loads((prev/'reports/final_status.json').read_text())
    req(prior.get('stage')=='R05' and prior['n01_manifest_sha256']==manifest_hash,'Previous R05 belongs to another dataset')
    for source in ['rotation','simclr']:
        req(prior['r04_provenance'][source]['encoder_sha256']==provenance[source]['encoder_sha256'],'Previous R05 encoder mismatch')
    copied=0
    for split in EXPECTED:
        pprev=prior['manifests'].get(split)
        if pprev:
            f=prev/pprev['file']
            if f.exists() and sha(f)==pprev['sha256']:
                d=OUT/pprev['file'];d.parent.mkdir(exist_ok=True,parents=True)
                if not d.exists():shutil.copy2(f,d);copied+=1
        for source in ['rotation','simclr']:
            item=prior['features'].get(split,{}).get(source)
            if item:
                f=prev/item['file']
                if f.exists() and sha(f)==item['sha256']:
                    d=OUT/item['file'];d.parent.mkdir(exist_ok=True,parents=True)
                    if not d.exists():shutil.copy2(f,d);copied+=1
    print('R05 resumed previously verified files:',copied)


R04 reconstruction rotation max probability difference 0.000306
R04 reconstruction simclr max probability difference 0.000733
Both feature encoders verified, classifier heads removed.


## Export immutable Rotation + SimCLR feature banks for train, validation, meta_train (never test)

In [5]:
loader_args={'batch_size':BATCH,'shuffle':False,'num_workers':WORKERS,'pin_memory':DEVICE.type=='cuda'}
if WORKERS:loader_args['prefetch_factor']=2
existing={}
for split,f in frames.items():
    manifest=OUT/'manifest'/f'{split}.csv'
    meta=f[['image_id','group_id','class_index','label','source_sha256','image_relpath']].copy()
    if manifest.exists():
        prior=pd.read_csv(manifest,dtype='string',keep_default_na=False)
        req(prior.equals(meta),f'Cannot reuse altered feature manifest {split}')
    else:meta.to_csv(manifest,index=False)
    files={src:OUT/'features'/f'{split}_{src}.npy' for src in ['rotation','simclr']}
    if all(p.exists() for p in files.values()):
        for src,p in files.items():
            arr=np.load(p,mmap_mode='r')
            req(arr.shape==(len(f),2048) and np.isfinite(arr).all(),f'Existing feature file corrupt: {p}')
        existing[split]='reused';print('REUSED',split);continue
    if any(p.exists() for p in files.values()):
        print('Partial feature export detected; rebuilding both for',split)
        for p in files.values():p.unlink(missing_ok=True)
    if time.monotonic()-START>MAX_HOURS*3600:print('Runtime budget reached before',split);break
    loader=DataLoader(Images(f),**loader_args)
    arrays={src:np.lib.format.open_memmap(str(p.with_suffix('.tmp.npy')),mode='w+',dtype='float32',shape=(len(f),2048)) for src,p in files.items()}
    offset=0
    with torch.inference_mode():
        for bi,x in enumerate(loader):
            if bi%32==0 and time.monotonic()-START>MAX_HOURS*3600:
                print('Runtime guard reached during',split,'— removing incomplete exports');break
            x=x.to(DEVICE,non_blocking=True)
            for src in ['rotation','simclr']:
                with torch.autocast(device_type=DEVICE.type,enabled=AMP and DEVICE.type=='cuda'):
                    z=models[src](x).float().cpu().numpy()
                req(z.shape==(len(x),2048) and np.isfinite(z).all(),f'{src}: Nonfinite features')
                arrays[src][offset:offset+len(x)]=z
            offset+=len(x)
    for a in arrays.values():a.flush()
    del arrays
    if offset!=len(f):
        for p in files.values():p.with_suffix('.tmp.npy').unlink(missing_ok=True)
        break
    for src,p in files.items():p.with_suffix('.tmp.npy').replace(p)
    existing[split]='generated';print('SAVED',split,'images=',len(f))

done=all(all((OUT/'features'/f'{split}_{src}.npy').exists() for src in ['rotation','simclr']) for split in EXPECTED)
report={'stage':'R05','name':NAME,'seed':SEED,'status':'COMPLETED' if done else 'PAUSED',
        'classes':CLASSES,'n01_manifest_sha256':manifest_hash,
        'r00_protocol_sha256':checks['configs/experiment_protocol.json'],
        'r04_provenance':provenance,'splits':EXPECTED,'feature_dim_per_source':2048,'test_touched':False,
        'method':'frozen R04 fine-tuned N01 ResNet50 encoders; deterministic, no-augmentation eval transform; feature bank',
        'features':{split:{src:{'file':f'features/{split}_{src}.npy','sha256':sha(OUT/'features'/f'{split}_{src}.npy')}
                           for src in ['rotation','simclr'] if (OUT/'features'/f'{split}_{src}.npy').exists()}
                    for split in EXPECTED},
        'manifests':{split:{'file':f'manifest/{split}.csv','sha256':sha(OUT/'manifest'/f'{split}.csv')} for split in EXPECTED},
        'seconds':round(time.monotonic()-START,2),'pytorch':torch.__version__}
jsonwrite(OUT/'reports/final_status.json',report)
print('R05',report['status'],'Output:',OUT)
if not done:print('R05 PAUSED; completed split feature files remain available. Attach saved R05 output for safe resume (see R05_PREVIOUS_ROOT).')


SAVED train images= 16490
SAVED validation images= 2186
SAVED meta_train images= 2267
R05 COMPLETED Output: /kaggle/working/lesion-r05-n01-dual-ssl-featurebank-s42
